In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(r"C:\Users\DELL\OneDrive\Desktop\techwork(3)(2)\Employee_Dataset.csv")
df

,employee_id,department,designation,age,salary,joining_date,last_promotion_date,experience_years,performance_rating,is_active
0,EMP1000,IT,Senior Analyst,60,85000,invalid,01/04/2021,5,3,NaN
1,emp_1,IT,Analyst,NaN,55000,10/06/2020,invalid,5,3,True
2,EMP1002,Sales,NaN,28,85000,NaN,2022-03-01,3,4,NaN
3,EMP1003,it,Analyst,45,35000,invalid,01/04/2021,1,3,NaN
4,NaN,IT,mgr,150,85000,NaN,invalid,12,2,True
...,...,...,...,...,...,...,...,...,...,...
995,EMP1995,Finance,NaN,60,55000,2019-05-10,2022-03-01,1,4,yes
996,EMP1996,IT,Analyst,35,250000,invalid,invalid,-2,5,NaN
997,EMP1997,Sales,Senior Analyst,150,250000,2021/07/15,invalid,3,excellent,yes
998,emp_998,Finance,NaN,60,85000,2019-05-10,NaN,12,NaN,True


In [3]:
#1.Convert joining_date to datetime and count how many rows failed conversion
df['joining_date'] = pd.to_datetime(df['joining_date'], errors='coerce')
failed_join_dates = df['joining_date'].isna().sum()
print("Failed joining_date conversions:", failed_join_dates)

Failed joining_date conversions: 377


C:\Users\DELL\AppData\Local\Temp\ipykernel_4656\293344179.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['joining_date'] = pd.to_datetime(df['joining_date'], errors='coerce')


In [4]:
#2.Clean employee_id and identify how many duplicate employees exist after standardization
df['employee_id'] = df['employee_id'].astype(str).str.strip().str.upper()
duplicate_employees = df['employee_id'].duplicated().sum()
print("Duplicate employees:", duplicate_employees)

Duplicate employees: 503


In [5]:
#3.Standardize department and calculate the average salary per department excluding invalid salaries
df['department'] = df['department'].str.strip().str.title() 
df['salary'] = pd.to_numeric(df['salary'], errors='coerce')
avg_salary_dept = df.groupby('department')['salary'].mean()
avg_salary_dept

department
Finance    113253.012048
Hr         114800.000000
It         107944.444444
Sales      119000.000000
Name: salary, dtype: float64

In [6]:
#4.Convert age to numeric and find employees with valid salary but invalid age
df['age'] = pd.to_numeric(df['age'], errors='coerce')
invalid_age_valid_salary = df[df['salary'].notna() & df['age'].isna()]
print(invalid_age_valid_salary[['employee_id', 'salary', 'age']])
len(invalid_age_valid_salary)

    employee_id    salary  age
1         EMP_1   55000.0  NaN
30      EMP1030   85000.0  NaN
31          NAN   55000.0  NaN
38       EMP_38   35000.0  NaN
45          NAN  120000.0  NaN
..          ...       ...  ...
959         NAN  120000.0  NaN
974     EMP_974   85000.0  NaN
976     EMP1976  250000.0  NaN
982         NAN   55000.0  NaN
991         NAN  120000.0  NaN

[120 rows x 3 columns]


120

In [7]:
#5.Clean salary and detect outliers using the IQR method
Q1 = df['salary'].quantile(0.25)
Q3 = df['salary'].quantile(0.75)
IQR = Q3 - Q1
salary_outliers = df[(df['salary'] < Q1 - 1.5*IQR) | (df['salary'] > Q3 + 1.5*IQR)]
salary_outliers[['employee_id', 'salary']]

,employee_id,salary
11,NAN,250000.0
16,NAN,250000.0
25,NAN,250000.0
34,NAN,250000.0
41,NAN,250000.0
...,...,...
955,NAN,250000.0
969,EMP1969,250000.0
976,EMP1976,250000.0
996,EMP1996,250000.0


In [8]:
#6.Convert performance_rating into numeric and calculate the median rating per designation
df['performance_rating'] = pd.to_numeric(df['performance_rating'], errors='coerce')
median_rating = df.groupby('designation')['performance_rating'].median()
median_rating

designation
ANALYST           3.0
Analyst           3.0
Manager           3.0
Senior Analyst    3.0
mgr               3.0
Name: performance_rating, dtype: float64

In [9]:
#7.Identify employees whose last_promotion_date is earlier than their joining_date
df['last_promotion_date'] = pd.to_datetime(df['last_promotion_date'], errors='coerce')
invalid_promo = df[df['last_promotion_date'] < df['joining_date']]
invalid_promo[['employee_id', 'joining_date', 'last_promotion_date']]

,employee_id,joining_date,last_promotion_date
24,EMP1024,2021-07-15,2021-01-04
40,EMP_40,2021-07-15,2021-01-04
44,EMP_44,2021-07-15,2021-01-04
47,NAN,2021-07-15,2021-01-04
71,EMP_71,2021-07-15,2021-01-04
82,EMP_82,2021-07-15,2021-01-04
85,NAN,2021-07-15,2021-01-04
124,NAN,2021-07-15,2021-01-04
138,EMP_138,2021-07-15,2021-01-04
169,EMP_169,2021-07-15,2021-01-04


In [10]:
#8.Clean experience_years and find mismatches where experience exceeds employee age
df['experience_years'] = pd.to_numeric(df['experience_years'], errors='coerce')
exp_age_mismatch = df[df['experience_years'] > df['age']]
exp_age_mismatch[['employee_id', 'age', 'experience_years']]

,employee_id,age,experience_years
10,EMP_10,-5.0,8.0
17,EMP_17,-5.0,3.0
23,EMP_23,-5.0,-2.0
43,NAN,-5.0,1.0
49,NAN,-5.0,5.0
...,...,...,...
965,NAN,-5.0,12.0
981,NAN,-5.0,-2.0
985,EMP_985,-5.0,-2.0
990,EMP_990,-5.0,12.0


In [11]:
#9.Standardize designation and count how many active employees are in each designation
df['is_active'] = df['is_active'].astype(str).str.lower().map({'yes': True, 'no': False})
active_designation = df[df['is_active'] == True]['designation'].value_counts()
active_designation

designation
Senior Analyst    58
mgr               39
Analyst           35
Manager           34
ANALYST           23
Name: count, dtype: int64

In [12]:
#10.Convert is_active to boolean and find inactive employees with recent promotions
recent_promo = df[(df['is_active'] == False) &(df['last_promotion_date'] > pd.Timestamp.today() - pd.DateOffset(years=2))]
recent_promo[['employee_id', 'last_promotion_date', 'is_active']]

,employee_id,last_promotion_date,is_active


In [13]:
#11.Calculate employee tenure in years and find those with tenure above the 90th percentile
df['tenure'] = (pd.Timestamp.today() - df['joining_date']).dt.days / 365
high_tenure = df[df['tenure'] > df['tenure'].quantile(0.9)]
high_tenure[['employee_id', 'joining_date', 'tenure']]

,employee_id,joining_date,tenure


In [14]:
#12.Identify departments where more than 25% of salary values are missing or invalid
invalid_salary_pct = df.groupby('department')['salary'].apply(lambda x: x.isna().mean())
problem_depts = invalid_salary_pct[invalid_salary_pct > 0.25]
problem_depts

department
Finance    0.366412
Hr         0.321267
It         0.340659
Sales      0.398340
Name: salary, dtype: float64

In [15]:
#13.Create a flag for employees with high performance (≥4) but below-median salary
median_salary = df['salary'].median()
df['high_perf_low_salary'] = (df['performance_rating'] >= 4) & (df['salary'] < median_salary)
df[df['high_perf_low_salary']][['employee_id', 'performance_rating', 'salary']]
#df['high_perf_low_salary']

,employee_id,performance_rating,salary
21,NAN,4.0,55000.0
29,NAN,4.0,55000.0
31,NAN,4.0,55000.0
66,NAN,4.0,35000.0
92,NAN,5.0,35000.0
118,NAN,4.0,35000.0
145,EMP1145,4.0,35000.0
149,EMP1149,5.0,35000.0
189,NAN,4.0,55000.0
208,EMP_208,5.0,55000.0


In [16]:
#14.Detect employees with no promotion date but more than 5 years of experience
no_promo_exp = df[df['last_promotion_date'].isna() & (df['experience_years'] > 5)]
no_promo_exp[['employee_id', 'experience_years', 'last_promotion_date']]

,employee_id,experience_years,last_promotion_date
4,NAN,12.0,NaT
5,NAN,8.0,NaT
6,NAN,12.0,NaT
10,EMP_10,8.0,NaT
12,EMP_12,8.0,NaT
...,...,...,...
977,EMP_977,8.0,NaT
978,EMP1978,8.0,NaT
993,EMP1993,12.0,NaT
994,EMP1994,12.0,NaT


In [17]:
#15.Build a validation rule to flag rows violating at least two business constraints
df['rule_age_missing'] = df['age'].isna()
df['rule_salary_missing'] = df['salary'].isna()
df['rule_join_date_missing'] = df['joining_date'].isna()
df['violations'] = df[['rule_age_missing', 'rule_salary_missing', 'rule_join_date_missing']].sum(axis=1)
flag_rows = df[df['violations'] >= 2]
flag_rows[['employee_id', 'violations']]

,employee_id,violations
5,NAN,2
12,EMP_12,2
17,EMP_17,2
18,EMP_18,3
27,EMP1027,2
...,...,...
973,NAN,2
983,EMP1983,2
984,NAN,2
985,EMP_985,2


In [18]:
#16.After cleaning dates, find employees whose promotion gap is less than 1 year
gap = (df['last_promotion_date'] - df['joining_date']).dt.days / 365
short_gap = df[gap < 1]
short_gap[['employee_id', 'joining_date', 'last_promotion_date']]

,employee_id,joining_date,last_promotion_date
9,EMP1009,2020-10-06,2021-01-04
24,EMP1024,2021-07-15,2021-01-04
40,EMP_40,2021-07-15,2021-01-04
44,EMP_44,2021-07-15,2021-01-04
47,NAN,2021-07-15,2021-01-04
...,...,...,...
943,EMP1943,2020-10-06,2021-01-04
945,NAN,2020-10-06,2021-01-04
966,NAN,2020-10-06,2021-01-04
974,EMP_974,2021-07-15,2021-01-04
